In [2]:
import pyranges as pr
import gzip
import pandas as pd
import numpy as np
from scipy.sparse import dok_matrix

from hicexplorer import hicConvertFormat

In [3]:
path_interactions = "/home/joachim/data/MHH/epigenetic_graph_analyzer/phic/GSM6505208_Md_CapHiC_Merged_All-Tr_2ndBioRep_Chicago_output_washU_text.txt.gz"
path_output_2d = "/home/joachim/data/MHH/epigenetic_graph_analyzer/phic/GSM6505208_Md_CapHiC_Merged_All-Tr_2ndBioRep_Chicago_output_washU_text.csv"
path_output_cool = "/home/joachim/data/MHH/epigenetic_graph_analyzer/phic/GSM6505208_Md_CapHiC_Merged_All-Tr_2ndBioRep_Chicago_output_washU_text.cool"
path_mm10_chromosome_size = "/home/joachim/data/MHH/epigenetic_graph_analyzer/mm10.chrom.sizes"
path_promoter = "/home/joachim/data/MHH/epigenetic_graph_analyzer/mm10_promoter.bed"


In [4]:
pd_interactions = pd.read_table(path_interactions, header=None, sep='\t|,', compression='gzip')

In [50]:
pd_interactions

,0,1,2,3,4,5,6
0,chr1,3100832,3102475,chr1,88220832,88222715,20.75
1,chr1,3100832,3102475,chr1,88239615,88243256,7.79
2,chr1,3100832,3102475,chr1,88265935,88274374,7.79
3,chr1,3100832,3102475,chr1,88278578,88280173,14.24
4,chr1,3100832,3102475,chr10,53085931,53086971,5.24
...,...,...,...,...,...,...,...
324460,chrY,1304307,1309275,chr9,8775476,8777570,7.28
324461,chrY,1304307,1309275,chrY,790667,792380,5.28
324462,chrY,1304307,1309275,chrY,981956,982979,7.06
324463,chrY,1304307,1309275,chrY,1017923,1018908,5.10


In [21]:
pd_interactions.to_csv(path_output_2d, index=False, header=False, sep='\t')

In [22]:
args = "--matrices {} --outFileName {} --inputFormat 2D-text --outputFormat cool -r 10000 --chromosomeSizes {}".format(path_output_2d, path_output_cool, path_mm10_chromosome_size).split()
hicConvertFormat.main(args)

ERROR:hicmatrix.HiCMatrix:Index error
Traceback (most recent call last):
  File "/home/joachim/miniconda3/envs/ega/lib/python3.10/site-packages/hicmatrix/HiCMatrix.py", line 262, in getRegionBinRange
    endbin = sorted(self.interval_trees[chrname][endpos:endpos + 1])[0].data
IndexError: list index out of range
ERROR:hicmatrix.HiCMatrix:Index error
Traceback (most recent call last):
  File "/home/joachim/miniconda3/envs/ega/lib/python3.10/site-packages/hicmatrix/HiCMatrix.py", line 262, in getRegionBinRange
    endbin = sorted(self.interval_trees[chrname][endpos:endpos + 1])[0].data
IndexError: list index out of range
ERROR:hicmatrix.HiCMatrix:Index error
Traceback (most recent call last):
  File "/home/joachim/miniconda3/envs/ega/lib/python3.10/site-packages/hicmatrix/HiCMatrix.py", line 262, in getRegionBinRange
    endbin = sorted(self.interval_trees[chrname][endpos:endpos + 1])[0].data
IndexError: list index out of range
ERROR:hicmatrix.HiCMatrix:Index error
Traceback (most recent 

In [5]:
import hicmatrix.HiCMatrix as hm
from hicexplorer import utilities
import scipy.sparse as sp


In [6]:
chic_matrix = hm.hiCMatrix(path_output_cool)

In [7]:
chic_matrix.matrix.shape[0]

273121

In [8]:
pyrange_promoter = pr.read_bed(path_promoter)

In [9]:
pyrange_promoter

,Chromosome,Start,End,Name,Score,Strand,ThickStart,ThickEnd
0,chr1,4807845,4807905,Lypla1_1,900,+,4807894,4807905
1,chr1,4857741,4857801,Tcea1_1,900,+,4857790,4857801
2,chr1,5083079,5083139,Atp6v1h_1,900,+,5083128,5083139
3,chr1,5588420,5588480,Oprk1_1,900,+,5588469,5588480
4,chr1,6214614,6214674,Rb1cc1_1,900,+,6214663,6214674
...,...,...,...,...,...,...,...,...
25106,chrY,1459766,1459826,Usp9y_1,900,-,1459766,1459777
25107,chrY,2170384,2170444,Zfy2_1,900,-,2170384,2170395
25108,chrY,3410158,3410218,Rbmy_1,900,-,3410158,3410169
25109,chrY,89713277,89713337,FJ541083_1,900,-,89713277,89713288


In [47]:
len(pyrange_promoter)

25111

In [10]:
matrix_list = []
bait_list = []
for index, row in pyrange_promoter.as_df().iterrows():
    bait_location = row[['Chromosome', 'Start', 'End']]
    bait_list.append(row[['Name']])
    view_point_start, view_point_end = chic_matrix.getRegionBinRange(bait_location[0], bait_location[1], bait_location[2])

    matrix_list.append(chic_matrix.matrix[view_point_start, :])

# Assuming you have a list of CSR matrices: matrix_list
sample_matrix = sp.vstack(matrix_list)


In [49]:
sample_matrix

<25111x273121 sparse matrix of type '<class 'numpy.float64'>'
	with 167208 stored elements in Compressed Sparse Row format>

In [11]:
import networkx as nx

import community as community_louvain


In [12]:
G = nx.Graph()

bait_ids, OE_ids = sample_matrix.nonzero()
for i, (bait, OE) in enumerate(zip(bait_ids, OE_ids)):
    if (bait != OE):
        G.add_weighted_edges_from([(bait, OE , sample_matrix.data[i])])
        G.nodes[OE]['colorType'] = "orange"


for bait in bait_ids:
    G.nodes[bait]['name'] = bait_list[bait][0]
    G.nodes[bait]['colorType'] = "green"

partition = community_louvain.best_partition(G)

for node, partition_id in partition.items():
    G.nodes[node]['partition'] = partition_id
    G.nodes[node]['partition_color'] = partition_id


nx.write_graphml(G, "graph.graphml")

In [1]:
partition

NameError: name 'partition' is not defined

In [61]:
max(partition.values())

1334

In [20]:
partition[8766]

6

In [18]:
for bait in bait_list:
    if "Hoxa" in bait[0]:
        print(bait)

Name    Hoxaas2_1
Name: 8138, dtype: object
Name    Hoxaas3_1
Name: 8140, dtype: object
Name    Hoxaas3_2
Name: 8141, dtype: object
Name    Hoxa11os_1
Name: 8142, dtype: object
Name    Hoxa11os_2
Name: 8143, dtype: object
Name    Hoxa1_1
Name: 8766, dtype: object
Name    Hoxa2_1
Name: 8767, dtype: object
Name    Hoxa4_1
Name: 8768, dtype: object
Name    Hoxa6_1
Name: 8769, dtype: object
Name    Hoxa9_1
Name: 8770, dtype: object
Name    Hoxa10_2
Name: 8772, dtype: object
Name    Hoxa10_1
Name: 8773, dtype: object
Name    Hoxa11_1
Name: 8774, dtype: object
Name    Hoxa13_1
Name: 8775, dtype: object


In [21]:
import matplotlib.colors as mcolors

In [22]:
mcolors.Colormap

matplotlib.colors.Colormap